# STAC data prep — Soil sand content, depth cube (Zarr)

Notebook to stack the six depth intervals of the SoilGrids 2.0 sand content (5 km, from Google Earth Engine) into **one Zarr datacube** with dimensions `(depth, y, x)`.

This is the **Zarr variant** of the dataset. The **COG variant** (six 2D COGs, published in GeoServer) is `28_soil_sand_cog.ipynb`; run both to compare them.

- **Why Zarr:** the z dimension (`depth`) is a real dimension, so a single read gives a depth profile or a depth slice (`ds.sel(depth=45)`) and the six layers travel as one object. A Zarr cube cannot be shown by GeoServer; it needs a client that reads Zarr (xarray, the frontend datacube viewer) or a tile service built on top.
- **Variable:** `sand` in %, `float32`, `NaN` = no data (about 53 % of the grid).
- **Dimensions:** `depth` = middle of each interval in cm (2.5, 10, 22.5, 45, 80, 150) with the coordinates `depth_top` and `depth_bottom` (cm) on the same axis; `y` and `x` in EPSG:4326.
- **Chunks:** `(1, 512, 512)`, one chunk per depth and tile, so a single-depth map or a window reads only what it needs.
- **Format:** Zarr v2 (`zarr_format=2`), as in `16_chloride.ipynb`.
- **Precision:** the sources are `float64`, the cube is `float32`. The check compares against the `float32` cast of the source.
- **Metadata template:** `metadata/metadata_Soil_Sand_Content.json`.

Sources (`P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian`): `sand_5km_<depth>.tif` for the depths 0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100-200cm.

**Running in docker:** the `P:` drive is not visible in the container, so run from the host:

```powershell
docker\run_notebook_with_p_drive.ps1 -Notebook 30_soil_sand_zarr.ipynb -Variable soil_sand_zarr -InputPattern "sand_5km_*.tif"
```

Run `scripts/30_soil_sand_zarr.ipynb` afterwards to build the STAC collection/item.

### Import packages

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import rasterio
import rioxarray  # noqa: F401  (registers ds.rio)
import xarray as xr

### Define drive paths

`DATA_VIVIAN` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.

In [2]:
COLLECTION_ID = "soil_sand_zarr"
VARIABLE = "sand"
UNITS = "%"
DEPTHS = ['0-5cm', '5-15cm', '15-30cm', '30-60cm', '60-100cm', '100-200cm']
# The maps are static (no time series). SoilGrids 2.0 was released in 2020, which is used as the validity period.
TEMPORAL_EXTENT = ['2020-01-01T00:00:00Z', '2020-12-31T23:59:59Z']
# The template says 'mass fraction in permille', but the 5 km maps from Google Earth Engine are in % (clay 0-68, sand 11-100)
DESCRIPTION_FIX = ("mass fraction in \u2030", "mass fraction in % (g/100 g)")

data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_tifs = {depth: data_vivian / f"{VARIABLE}_5km_{depth}.tif" for depth in DEPTHS}
metadata_template_path = stac_repo / "metadata" / "metadata_Soil_Sand_Content.json"
if not metadata_template_path.is_file():  # same file as in the P: metadata folder
    metadata_template_path = data_vivian / "metadata" / "metadata_Soil_Sand_Content.json"

output_dir = data_vivian / "stac" / COLLECTION_ID
output_zarr = output_dir / f"{COLLECTION_ID}.zarr"
metadata_output_path = output_dir / f"metadata_{COLLECTION_ID}.json"
output_dir.mkdir(parents=True, exist_ok=True)

for p in (*source_tifs.values(), metadata_template_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print("Sources:", *source_tifs.values(), sep="\n  ")
print("Metadata template:", metadata_template_path)
print("Output Zarr:", output_zarr)

Sources:
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_0-5cm.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_5-15cm.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_15-30cm.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_30-60cm.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_60-100cm.tif
  P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\sand_5km_100-200cm.tif
Metadata template: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\metadata\metadata_Soil_Sand_Content.json
Output Zarr: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_sand_zarr\soil_sand_zarr.zarr


### Read raw data

In [3]:
layers, crs, transform, shape = {}, None, None, None
for depth, path in source_tifs.items():
    with rasterio.open(path) as src:
        print(f"{path.name}: {src.width} x {src.height} | dtype {src.dtypes[0]} | nodata {src.nodata} | crs {src.crs.to_epsg()}")
        if shape is None:
            crs, transform, shape = src.crs, src.transform, src.shape
        # All depths must be on the same grid to be stacked
        assert (src.crs, src.transform, src.shape) == (crs, transform, shape), f"{path.name} is on a different grid"
        layers[depth] = src.read(1)
print("Common grid:", shape, "|", tuple(round(v, 6) for v in transform)[:6])

sand_5km_0-5cm.tif: 1671 x 2076 | dtype float64 | nodata None | crs 4326
sand_5km_5-15cm.tif: 1671 x 2076 | dtype float64 | nodata None | crs 4326
sand_5km_15-30cm.tif: 1671 x 2076 | dtype float64 | nodata None | crs 4326
sand_5km_30-60cm.tif: 1671 x 2076 | dtype float64 | nodata None | crs 4326
sand_5km_60-100cm.tif: 1671 x 2076 | dtype float64 | nodata None | crs 4326
sand_5km_100-200cm.tif: 1671 x 2076 | dtype float64 | nodata None | crs 4326
Common grid: (2076, 1671) | (0.044916, 0.0, -20.032431, 0.0, -0.044916, 46.622563)


### Make CF compliant alterations (dataset dependent)

- Build the `x`, `y` coordinates from the grid (pixel centres) and the `depth` axis with top/bottom coordinates and `units`, `positive="down"`.
- Cast `float64` to `float32`; no nodata value is needed in Zarr, empty cells stay `NaN` (`_FillValue = NaN`).
- Fill the metadata fields that are empty in the template: `SPATIAL_EXTENT`, `SPATIAL_RESOLUTION`, `NODATA`, `DATA_TYPE`, `DIMENSIONS`, `MEDIA_TYPE`, `COLLECTION_ID`, `UNITS`, ...

In [4]:
tops = np.array([int(d.removesuffix("cm").split("-")[0]) for d in DEPTHS])
bottoms = np.array([int(d.removesuffix("cm").split("-")[1]) for d in DEPTHS])
depth_mid = (tops + bottoms) / 2

ny, nx = shape
x = transform.c + (np.arange(nx) + 0.5) * transform.a
y = transform.f + (np.arange(ny) + 0.5) * transform.e

data = np.stack([layers[d] for d in DEPTHS]).astype("float32")
ds = xr.Dataset(
    {VARIABLE: (("depth", "y", "x"), data, {"long_name": "Sand content (mean per depth interval)", "units": UNITS})},
    coords={
        "depth": ("depth", depth_mid, {"long_name": "middle of the depth interval", "units": "cm", "positive": "down", "axis": "Z"}),
        "depth_top": ("depth", tops, {"long_name": "top of the depth interval", "units": "cm"}),
        "depth_bottom": ("depth", bottoms, {"long_name": "bottom of the depth interval", "units": "cm"}),
        "y": ("y", y, {"standard_name": "latitude", "units": "degrees_north", "axis": "Y"}),
        "x": ("x", x, {"standard_name": "longitude", "units": "degrees_east", "axis": "X"}),
    },
)
ds = ds.rio.write_crs(4326)
ds[VARIABLE].encoding = {"chunks": (1, 512, 512), "_FillValue": np.nan}

with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

left, top = transform.c, transform.f
right, bottom = left + nx * transform.a, top + ny * transform.e
metadata.update({
    "TITLE_ABBREVIATION": "SoilSandZarr",
    "DESCRIPTION": metadata["DESCRIPTION"].replace(*DESCRIPTION_FIX),
    "HISTORY": [
        "SoilGrids 2.0 sand content (ISRIC), mean per depth interval, downloaded at 5 x 5 km from Google Earth Engine",
        "Six standard depth intervals (0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100-200cm) stacked into one (depth, y, x) Zarr datacube (float32, chunks 1 x 512 x 512)",
    ],
    "MEDIA_TYPE": "application/vnd+zarr",
    "DATA_MODEL": "datacube_zarr",
    "DIMENSIONS": ["x", "y", "depth"],
    "SPATIAL_EXTENT": [round(max(left, -180.0), 6), round(max(bottom, -90.0), 6), round(min(right, 180.0), 6), round(min(top, 90.0), 6)],
    "TEMPORAL_EXTENT": TEMPORAL_EXTENT,
    "KEYWORDS": ['sand content', 'soil texture', 'soil', 'SoilGrids', 'depth intervals'] + ["zarr", "datacube"],
    "TAGS": ['soil', 'sand', 'texture', 'zarr'],
    "LONG_NAME": "Sand content (mean per depth interval)",
    "UNITS": UNITS,
    "COMMENT": (
        "Datacube with six depth intervals (0-5cm, 5-15cm, 15-30cm, 30-60cm, 60-100cm, 100-200cm); the depth coordinate is the middle of the interval in cm, "
        "depth_top and depth_bottom hold the interval limits. Values are the mean sand content in % (g/100 g), predicted by "
        "SoilGrids 2.0 and aggregated to a 5 km grid via Google Earth Engine. NaN means no data. Static map, not a time series."
    ),
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": round(abs(transform.a), 4),
    "NODATA": None,
    "DATA_TYPE": str(ds[VARIABLE].dtype),
    "COLLECTION_ID": COLLECTION_ID,
})

print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("SPATIAL_RESOLUTION:", metadata["SPATIAL_RESOLUTION"])
print(ds)

SPATIAL_EXTENT: [-20.032431, -46.622563, 55.021811, 46.622563]
SPATIAL_RESOLUTION: 0.0449
<xarray.Dataset> Size: 83MB
Dimensions:       (depth: 6, y: 2076, x: 1671)
Coordinates:
  * depth         (depth) float64 48B 2.5 10.0 22.5 45.0 80.0 150.0
    depth_top     (depth) int64 48B 0 5 15 30 60 100
    depth_bottom  (depth) int64 48B 5 15 30 60 100 200
  * y             (y) float64 17kB 46.6 46.56 46.51 ... -46.51 -46.56 -46.6
  * x             (x) float64 13kB -20.01 -19.97 -19.92 ... 54.91 54.95 55.0
    spatial_ref   int64 8B 0
Data variables:
    sand          (depth, y, x) float32 83MB nan nan nan nan ... nan nan nan nan


## Write the Zarr store and the metadata

In [5]:
# Zarr attributes must be JSON-serialisable; lists and dicts are kept as JSON text like in the COG notebooks
attrs = {k: (json.dumps(v) if isinstance(v, (list, dict)) else v) for k, v in metadata.items() if v is not None}
attrs["Conventions"] = "CF-1.8"
ds.attrs.update(attrs)

# zarr_format=2 keeps the output readable by the repo's "coclico" conda env (zarr-python 2.x)
ds.to_zarr(output_zarr, mode="w", zarr_format=2)

with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)
print("Wrote Zarr store:", output_zarr)
print("Wrote metadata:", metadata_output_path)

Wrote Zarr store: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_sand_zarr\soil_sand_zarr.zarr
Wrote metadata: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\soil_sand_zarr\metadata_soil_sand_zarr.json


### Verify the round trip

The stored cube must equal the `float32` cast of every source GeoTIFF, depth by depth, including the `NaN` positions.

In [6]:
check = xr.open_zarr(output_zarr, consolidated=False)
assert check[VARIABLE].dims == ("depth", "y", "x")
assert check[VARIABLE].dtype == np.float32
assert check.attrs["COLLECTION_ID"] == COLLECTION_ID
assert check[VARIABLE].encoding["chunks"] == (1, 512, 512)

for i, depth in enumerate(DEPTHS):
    new = check[VARIABLE].isel(depth=i).values
    old = layers[depth].astype("float32")
    assert np.array_equal(np.isnan(old), np.isnan(new)), f"NaN mask changed for {depth}"
    assert np.array_equal(old[~np.isnan(old)], new[~np.isnan(new)]), f"Values changed for {depth}"
    print(f"{depth}: identical | valid {(~np.isnan(new)).sum()} of {new.size}")

# z-slice: one depth (middle of the 30-60 cm interval) and a depth profile at one point
print(check[VARIABLE].sel(depth=45.0).shape, check[VARIABLE].sel(x=31.0, y=30.5, method="nearest").values)
print("OK: soil sand Zarr store and metadata are ready for the STAC builder notebook.")

0-5cm: identical | valid 1617450 of 3468996
5-15cm: identical | valid 1617450 of 3468996
15-30cm: identical | valid 1617450 of 3468996
30-60cm: identical | valid 1617450 of 3468996
60-100cm: identical | valid 1617450 of 3468996
100-200cm: identical | valid 1617450 of 3468996
(2076, 1671) [38.011883 37.62104  36.525005 36.11798  36.32156  39.04222 ]
OK: soil sand Zarr store and metadata are ready for the STAC builder notebook.
